In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path
plt.style.use('seaborn-v0_8-whitegrid')
Path('artifacts').mkdir(exist_ok=True)
print('Imports OK')

In [ ]:
results = pd.DataFrame([
    {'Model': 'SASRec',     'HR@5': 0.2272, 'HR@10': 0.3328, 'HR@20': 0.4449, 'NDCG@10': 0.1873, 'Role': 'Tier 3'},
    {'Model': 'SVD f=128',  'HR@5': 0.0535, 'HR@10': 0.1020, 'HR@20': 0.1642, 'NDCG@10': 0.0487, 'Role': 'Tier 2'},
    {'Model': 'UserUserCF', 'HR@5': 0.0470, 'HR@10': 0.0830, 'HR@20': 0.1460, 'NDCG@10': 0.0400, 'Role': 'Tier 2'},
    {'Model': 'ItemItemCF', 'HR@5': 0.0460, 'HR@10': 0.0816, 'HR@20': 0.1340, 'NDCG@10': 0.0390, 'Role': 'Tier 2'},
    {'Model': 'Two-Tower',  'HR@5': 0.0353, 'HR@10': 0.0647, 'HR@20': 0.1078, 'NDCG@10': 0.0315, 'Role': 'Tier 3'},
    {'Model': 'BERT4Rec',   'HR@5': 0.0210, 'HR@10': 0.0417, 'HR@20': 0.0720, 'NDCG@10': 0.0199, 'Role': 'Experimental'},
])
print(results.to_string(index=False))

In [ ]:
colors = {'Tier 3': 'steelblue', 'Tier 2': 'coral', 'Experimental': 'lightgray'}
fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(results['Model'], results['HR@10'],
               color=[colors[r] for r in results['Role']], edgecolor='white')
ax.axvline(0.1, color='gray', linestyle='--', linewidth=0.8, label='0.10 reference')
ax.set_xlabel('HR@10 (full-catalog evaluation)')
ax.set_title('Model Comparison \u2014 Hit Rate @ 10 (MovieLens 1M)')
for bar, val in zip(bars, results['HR@10']):
    ax.text(val + 0.003, bar.get_y() + bar.get_height()/2,
            f'{val:.4f}', va='center', fontsize=9)
patches = [mpatches.Patch(color=v, label=k) for k, v in colors.items()]
ax.legend(handles=patches)
plt.tight_layout()
plt.savefig('artifacts/model_comparison_hr10.png', dpi=120)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
for _, row in results.iterrows():
    ax.scatter(row['HR@10'], row['NDCG@10'], s=80,
               color=colors[row['Role']], zorder=3)
    ax.annotate(row['Model'], (row['HR@10'], row['NDCG@10']),
                textcoords='offset points', xytext=(6, 3), fontsize=9)
ax.set_xlabel('HR@10'); ax.set_ylabel('NDCG@10')
ax.set_title('HR@10 vs NDCG@10 \u2014 All Models')
patches = [mpatches.Patch(color=v, label=k) for k, v in colors.items()]
ax.legend(handles=patches)
plt.tight_layout()
plt.savefig('artifacts/hr_vs_ndcg.png', dpi=120)
plt.show()

In [ ]:
epochs = [1, 3, 5, 10, 15, 20, 26, 31]
hr10   = [0.0356, 0.1174, 0.2091, 0.2836, 0.3104, 0.3224, 0.3328, 0.3305]
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(epochs, hr10, 'o-', color='steelblue', linewidth=2, markersize=6)
ax.axhline(0.1020, color='coral', linestyle='--', linewidth=1, label='SVD baseline (0.1020)')
ax.axhline(0.0830, color='gray',  linestyle='--', linewidth=1, label='CF baseline (0.0830)')
ax.scatter([26], [0.3328], s=150, color='red', zorder=5, label='Best epoch 26 (0.3328)')
ax.set_xlabel('Epoch'); ax.set_ylabel('HR@10 (full-catalog)')
ax.set_title('SASRec Training Curve \u2014 MovieLens 1M')
ax.legend()
plt.tight_layout()
plt.savefig('artifacts/sasrec_learning_curve.png', dpi=120)
plt.show()

In [ ]:
results['Delta vs SVD'] = (results['HR@10'] - 0.1020).map(lambda x: f'{x:+.4f}')
results['3x SVD?'] = results['HR@10'].map(lambda x: 'YES' if x > 0.306 else '')
print(results[['Model','HR@10','NDCG@10','Delta vs SVD','3x SVD?']].to_string(index=False))
print()
print('Conclusion: SASRec HR@10=0.3328 is 3.26x better than SVD (0.1020)')